# RT-DETR building detection: download, tile, zero-shot, fine-tune and infer

One notebook for the whole pipeline. It runs on **Google Colab**, **Kaggle** or your own machine, and everything runs through **uv**.

| Step | What it does | Command behind it |
|---|---|---|
| 1 | Set up the folder (clone from GitHub on Colab/Kaggle) and install with uv | `uv sync` |
| 2 | Download an RT-DETR checkpoint (any variant from the hub) | `download_model.py` |
| 3 | Download the COGs listed in `list_of_ids.txt` | `download_images.py --list list_of_ids.txt` |
| 4 | Resample and tile the scenes (optional, for labelling) | `tile_cogs.py` |
| 5 | Fine-tune RT-DETR with PyTorch Lightning, configured by Hydra | `train.py` |
| 6 | Detect buildings over the COGs with rasterio windows and write GeoJSON | `infer.py` |

**Settings you can change are in the cells below.** Everything else is controlled by the YAML files in `configs/` and can be overridden on the command line, for example `train.freeze_backbone=true`.

## 1. Set up

On Colab or Kaggle this clones the repository. Set `REPO_URL` to your GitHub repository. On your own machine it just uses the folder you have.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that the uv venv cannot load
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/emanuel-gf/FTL-Model-Training.git"   # set this to your repository
REPO_DIR = "webinar-DL"

if Path("pyproject.toml").exists() and Path("src/model.py").exists():
    pass                                                  # already inside colab_training/
elif Path("colab_training/pyproject.toml").exists():
    os.chdir("colab_training")
else:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(Path(REPO_DIR) / "colab_training")

print("working folder:", Path.cwd())

In [ ]:
!pip install -q uv
!nvidia-smi -L || echo "no GPU found: training will run on CPU (much slower)"
!uv --version

In [ ]:
!uv sync

## 2. Download an RT-DETR checkpoint

`PekingU/rtdetr_r50vd` is trained on COCO. Any RT-DETR on the Hugging Face hub works, for example `PekingU/rtdetr_r101vd`. The download is saved to `models/` so it can be reused offline.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"   # Colab sets an inline backend that the uv venv cannot load

HUB_ID = "PekingU/rtdetr_r50vd"
MODEL_DIR = "models/rtdetr_r50"

!uv run python download_model.py --hub {HUB_ID} --out {MODEL_DIR}

## 3. Download the COGs

The item IDs are in `list_of_ids.txt`. The COGs go to `cog/` and a `manifest.csv` records the licence and producer of each one.

In [ ]:
!uv run python download_images.py --list list_of_ids.txt --out cog

import pandas as pd
manifest = pd.read_csv("cog/manifest.csv")
manifest[["item_id", "status", "size_mb", "gsd_m", "producer", "license"]]

## 4. Resample and tile (for labelling)

Training and inference must use the **same ground resolution**. The project uses 10 cm/px. The tiles from this step are the images to annotate in labelme. Annotate them, then put the `.tif` tiles and their `.json` files in `data/labelme/images/`.

Set `DO_TILING = False` if you already have labelled tiles.

In [ ]:
DO_TILING = True
GSD = 0.10          # m/px used for training and inference
TILE = 640

if DO_TILING:
    !uv run python tile_cogs.py --input cog --gsd {GSD} --tile {TILE} --overlap 128 --output data/tiles --overwrite


## 5. Tile at 10 cm and export the annotations

The labelled tiles of step 4 are also at 10 cm, so the tiles use the same grid and resolution as the annotations. The dataset and the zero-shot tests use **10 cm, 640 × 640** tiles, so the COGs are tiled again into `data/tiles_10cm/tiles/`.

The labelme annotations of the labelled tiles (`data/labelme/images/*.tif` + `.json`) are clipped to these tiles and saved as one GeoJSON per annotated tile in `data/tiles_10cm/labels/`, with the same file names as the tiles. A tile is annotated only when it lies fully inside an annotated labelled tile. Only buildings are kept (`building` and `building-ext` are merged, `lote` is dropped).

In [ ]:
from pathlib import Path
from src.tiler import tile_cogs
from src.annotations import export_tile_annotations

GSD, TILE, OVERLAP = 0.10, 640, 128
RETILE = True                                       # False: keep the tiles already in data/tiles_10cm

if RETILE:
    tile_cogs("cog", "data/tiles_10cm", gsd=GSD, tile=TILE, overlap=OVERLAP,
              resampled_dir="cog/resampled")

export_tile_annotations("data/tiles_10cm/tiles_index.csv", labels_dir="data/labelme/images",
                        out_dir="data/tiles_10cm/labels", tile=TILE)
n_annotated = len(list(Path("data/tiles_10cm/labels").glob("*.geojson")))
print(f"{n_annotated} annotated tiles")
if n_annotated == 0:
    raise SystemExit("No annotated tiles: add the labelme .tif + .json pairs to data/labelme/images/")

## 6. FiftyOne dataset (10 cm tiles)

`src/fo_dataset.py` adds every tile of `data/tiles_10cm/` to a FiftyOne dataset named `building_tiles_10cm`. The ground truth is read from the GeoJSON of step 5 and stored in the field `ground_truth`. Unannotated tiles keep an empty ground truth and `annotated=False`.

In [ ]:
from src.fo_dataset import build_dataset, dataset_summary

dataset = build_dataset(tiles_dir="data/tiles_10cm", labels_dir="data/tiles_10cm/labels",
                        retile=False, overwrite=True)      # tiles and labels come from step 5
dataset_summary(dataset)

## 7. Zero-shot test (no fine-tuning)

Tag a few annotated tiles as `test`, then run the pretrained models on them and score the boxes against `ground_truth` (IoU 0.5, mAP over IoU 0.5:0.95). The default runs are **SAM2** and **OWLv2**.

- **sam2**: automatic masks, class-agnostic, kept by score and footprint area.
- **owlv2**: open-vocabulary detector, prompted with the text queries.
- **rtdetr** (in the list, not run by default): COCO RT-DETR has no building class, so every COCO detection counts as a building. This is a baseline.

In [ ]:
from src.fo_dataset import tag_test_samples

TEST_TILES = 10                                   # how many annotated tiles to tag as "test"
test_view = tag_test_samples(dataset, n=TEST_TILES, seed=51, tag="test")
# import fiftyone as fo; session = fo.launch_app(test_view)   # optional: browse the test tiles and their labels

In [ ]:
import torch
from src.zeroshot import pick_device, load_detector, run_field, run_detector, evaluate_run, compare_runs

DEVICE = pick_device()
print("device:", DEVICE)

# Edit this list to change the models or the weights.
#   kind:    sam2 | owlv2 | rtdetr
#   weights: a Hugging Face hub id, or a local folder (e.g. outputs/<run>/hf_model after fine-tuning)
ZERO_SHOT_RUNS = [
    dict(kind="sam2", weights="facebook/sam2.1-hiera-large", score_threshold=0.85, min_area_m2=10, max_area_m2=1500),
    dict(kind="owlv2", weights="google/owlv2-base-patch16-ensemble", score_threshold=0.2, queries=["a building"]),
    dict(kind="rtdetr", weights="PekingU/rtdetr_r50vd", score_threshold=0.3),
]
ACTIVE = [0, 1]                                   # indices of ZERO_SHOT_RUNS to run now: 0 = SAM2, 1 = OWLv2

In [ ]:
test_view = dataset.match_tags("test")
summary = {}
for i in ACTIVE:
    options = dict(ZERO_SHOT_RUNS[i])
    kind, weights = options.pop("kind"), options.pop("weights")
    field = run_field(kind, weights)
    print(f"\n== {kind}: {weights} -> field '{field}'")
    detector = load_detector(kind, weights, DEVICE, **options)
    run_detector(test_view, detector, field)
    summary[field] = evaluate_run(test_view, field)
    del detector                                  # free the GPU before the next model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

results_table = compare_runs(summary)

## 8. Batch prediction over the scenes

After the tests, one model is run over **every tile of the chosen scenes** and the detections are merged into one GeoJSON per scene in `outputs/batch/<model>/` (WGS84, with the score and the tile of each building). Overlapping tiles are merged with NMS at IoU 0.5.

`BATCH_RUN` is the index of the model in `ZERO_SHOT_RUNS`. `BATCH_SCENES = None` runs all scenes. Pass a list of scene ids to run a subset.

In [ ]:
from src.zeroshot import load_detector, predict_scenes, run_field

BATCH_RUN = 0                                      # index in ZERO_SHOT_RUNS: 0 = SAM2, 1 = OWLv2
BATCH_SCENES = None                                # None = every scene, or e.g. ["68a4a953e50784296730bf0f"]

options = dict(ZERO_SHOT_RUNS[BATCH_RUN])
kind, weights = options.pop("kind"), options.pop("weights")
out_dir = f"outputs/batch/{run_field(kind, weights)}"
detector = load_detector(kind, weights, DEVICE, **options)
counts = predict_scenes(detector, "data/tiles_10cm/tiles_index.csv", out_dir, scenes=BATCH_SCENES)
del detector
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(f"\nbuildings per scene -> {out_dir}/:", counts)

## 9. Fine-tune

Training settings live in `configs/train/finetune.yaml` (batch size, learning rate, epochs, freezing). Change them here with overrides, for example:

- `train.freeze_backbone=true` keeps the ResNet backbone fixed
- `train.freeze_decoder_layers=2` freezes the first two decoder layers
- `train.batch_size=2 train.accumulate_grad_batches=4` for a small GPU
- `train.epochs=10` for a quick run

The run writes its checkpoints, the split and the metrics to `outputs/<timestamp>/`. The test set is evaluated at the end, and the best checkpoint is exported to `hf_model/`.

In [ ]:
EPOCHS = 10
OVERRIDES = [
    f"train.epochs={EPOCHS}",
    "train.batch_size=4",
    f"model.checkpoint={MODEL_DIR}",
    # "train.freeze_backbone=true",
    # "train.freeze_decoder_layers=2",
]
!uv run python train.py {" ".join(OVERRIDES)}

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

runs = sorted(Path("outputs").glob("*/logs/version_*/metrics.csv"))
if not runs:
    raise SystemExit("No training metrics found yet: run the training cell first.")
metrics = pd.read_csv(runs[-1])
print("metrics:", runs[-1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col in [c for c in metrics.columns if c.endswith("loss") and metrics[c].notna().any()]:
    s = metrics[["epoch", col]].dropna()
    axes[0].plot(s["epoch"], s[col], marker="o", label=col)
axes[0].set_title("loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
for col in [c for c in metrics.columns if "map" in c and metrics[c].notna().any()]:
    s = metrics[["epoch", col]].dropna()
    axes[1].plot(s["epoch"], s[col], marker="o", label=col)
axes[1].set_title("COCO mAP")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## 10. Infer over the COGs

`infer.py` reads 640 × 640 windows from each COG with rasterio, predicts on them through Lightning, merges the overlapping detections and writes one GeoJSON per class (lon/lat, WGS84) to `outputs/geojson/`.

The model is the one exported by the last training run, unless you set `MODEL_CHECKPOINT` to another folder or checkpoint.

In [ ]:
exported = sorted(Path("outputs").glob("*/hf_model"))
MODEL_CHECKPOINT = str(exported[-1]) if exported else MODEL_DIR
print("model:", MODEL_CHECKPOINT)

COG_FILE = "cog/68a4a953e50784296730bf0f.tif"       # one scene; set to None to run every COG in cog/
SCORE_THRESHOLD = 0.4
infer_overrides = [
    f"model.checkpoint={MODEL_CHECKPOINT}",
    f"inference.score_threshold={SCORE_THRESHOLD}",
    f"inference.gsd={GSD}",
]
if COG_FILE:
    infer_overrides.append(f"inference.cog={COG_FILE}")
!uv run python infer.py {" ".join(infer_overrides)}

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

files = sorted(Path("outputs/geojson").rglob("*.geojson"))
print(f"{len(files)} GeoJSON files")
fig, ax = plt.subplots(figsize=(9, 9))
for f in files:
    fc = json.loads(f.read_text())
    for feat in fc["features"]:
        ring = feat["geometry"]["coordinates"][0]
        xs, ys = zip(*ring)
        ax.fill(xs, ys, alpha=0.35, edgecolor="k", linewidth=0.3)
    print(f"  {f.name}: {len(fc['features'])} features")
ax.set_aspect("equal")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Predicted buildings (lon/lat)")
plt.show()

## 11. Next steps

- Label more tiles in labelme and re-run from step 5. The more scenes there are, the better the split by scene works.
- Compare variants: change `MODEL_DIR` and `HUB_ID` in step 2 (for example `PekingU/rtdetr_r101vd`).
- Lower `SCORE_THRESHOLD` for more recall, raise it for more precision.
- The GeoJSON files open directly in QGIS.